In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<img src="./images/nvmath_head_panel@0.5x.png" alt="nvmath-python" />

# Getting Started with nvmath-python: Multi-GPU Multi-Node (MGMN) Execution

## Exercise: Distributed Matrix Multiplication

As an exercise, write a simple example that does a matrix multiplication using the distributed APIs (for multi-GPU execution).

**NOTE: This code is meant to be run inside the [08_distributed_apis.ipynb](./08_distributed_apis.ipynb) notebook.**

In [ ]:
%%mgmn

import cupy as cp
import nvmath.distributed
from nvmath.distributed.distribution import Slab 

# Global M, N, K
M, K, N = 4096, 1024, 512

# TN operation layout with A and B column-wise and the result row-wise
# uses AllGather+GEMM.
a_distribution = Slab.Y
b_distribution = Slab.Y
result_distribution = Slab.X
# Distributed matmul API needs to know how the operands are distributed.
distributions = [a_distribution, b_distribution, result_distribution]

# TN operation layout.
a_local_shape = a_distribution.shape(process_group.rank, (K, M))
b_local_shape = b_distribution.shape(process_group.rank, (K, N))

# Allocate input matrices A and B.
# cuBLASMp uses NCCL and doesn't require operands on NVSHMEM symmetric heap.
with cp.cuda.Device(device):
    a = cp.random.randn(*a_local_shape).astype(cp.float32)
    b = cp.random.randn(*b_local_shape).astype(cp.float32)

# cuBLASMp requires input matrices with Fortran (column-major) memory layout.
# For simplicity, we copy them to Fortran-ordered arrays.
a = cp.asfortranarray(a)
b = cp.asfortranarray(b)

import numpy as np
from nvmath.distributed.linalg.advanced import matrix_qualifiers_dtype

qualifiers = np.zeros((3,), dtype=matrix_qualifiers_dtype)
qualifiers[0]["is_transpose"] = True  # A is transposed

with nvmath.distributed.linalg.advanced.Matmul(
    a,
    b,
    distributions=distributions,
    qualifiers=qualifiers,
) as mm:
    mm.plan()
    result = mm.execute()

print(f"Local result shape: {result.shape}, dtype: {result.dtype}")